# 嵌入和向量表示

文本是离散的，数学是连续的。

## 问题描述

人类语言有几十种方式来表达同一件事情。关键词搜索将每个词当成一个独立的符号，在此时排不上用场。它搞不清楚“拒绝”和“反对”是同一个概念。

你需要一种表示方法，由文本的实际含义而非拼写决定相似度。

这种表示方法就是嵌入。

## 基本概念

### 什么是嵌入

嵌入就是使用一组稠密的浮点数组来表示文本。“稠密”的意思是每个维度都承载了信息，而不像词包、TF-IDF那样大部分维度都是零。

对于嵌入结果，将其表示为向量，直接观察和比较。

### Word2Vec 的突破

Word2Vec 是在一个旨在拉近词和它的近义词的神经网络上训练出来。最著名的结果就是：
```
king - man + woman = queen
```
嵌入向量间的数学计算捕捉到了语义关系，联立了几何运算与嵌入语义。

Word2Vec 的嵌入维度为300，每个词都能分到一个向量，不管它所处的上下文，这也成为了制约嵌入发展的主要原因。

### 从词到句子

词嵌入针对的是单个词元，生产系统中往往需要针对一整个句子、段落甚至文章进行嵌入。四种尝试：

#### 平均

池化所有的词嵌入。问题在于顺序被完全丢失了。“A打B”和“B打A”得到的结果一样。

#### CLS 词元

BERT会输出一个特殊的词元，即[CLS]用于分类。借鉴这种思想。问题在于这个词元训练出来是为了继续做生成而非相似度比较。

#### 对比学习

训练模型将相似的推到一起，不相似的拉远。基于上述的Sentence-BERT成为了现代嵌入模型的基础。

#### 指令调整的嵌入

先接受一个任务前提（比如关键词搜索，还是文档检索），然后模型根据对应任务产生嵌入向量。这使得一个模型可以同时执行多种工作。

### 相似度度量

给定两个嵌入向量，有三种方式衡量它们有多相似。

|度量|什么时候用|什么时候不用|
|---|---|---|
|余弦相似度|文本长度不同；大部分检索任务|模量会携带信息|
|点积|嵌入已经归一化了；速度优先|嵌入向量模量不同|
|欧式距离|聚类；空间最近邻问题|文章的长度相差很大|

### 向量数据库

主要是检索加速问题。ANN等...

### 切片策略

文档对于嵌入来说太长了，一个50页的PDF可能有好几十个主题，所以需要对文档进行切片，包括：
- 定长分块。 每N个Token切一刀，适合无结构化的文本，日志
- 句子分块。 每个句子分界处切一刀，适合文章，邮件等
- 递归分块。 按章写、段落从上往下切，适合Markdown，HTML等
- 语义分块。 嵌入语义变化明显时切分，有最好的检索质量

### 双编码和交叉编码（Bi-Encoder vs Cross-Encoder）

双解码对查询和文档独立编码，然后比较相似性。比较快，因为查询只编码一次。

交叉编码将查询和文档凭借后做编码，输出一个关联得分。比较慢，需要模型依次处理所有的对，不过更加准确。

生成上一般采用双编码先检索100个候选，然后用交叉编码挑选得分最高的10个。这就是 先检索再重排 流水线。

### 二值编码

对于一个N纬的向量，拿float精度存下去需要4*N bytes。

如果用一个bit位表示每个维上是正值还是负值，就只需要N/16 bytes。

用于初筛。牺牲大约5%的准确度，换32倍少的内存占用。